# Healthcare Fraud Detection — Data Audit

## Purpose

This notebook performs the initial audit of the healthcare fraud detection dataset.

We will examine:

- Dataset dimensions
- Column names and data types
- Missing values
- Duplicate records
- Target-label distribution
- Basic numerical summaries

This notebook does not perform modeling.

In [11]:
%pip install pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
from pathlib import Path

print("Pandas version:", pd.__version__)
print("Working folder:", Path.cwd())

Pandas version: 3.1.0
Working folder: /Users/aryamanmehta/Desktop/DatascienceMasters/Comp641-Fund DS/comp641-healthcare-fraud-project-fa2026/notebooks


In [13]:
data_folder = Path.cwd().parent / "data"

csv_files = sorted(data_folder.rglob("*.csv"))

print("CSV files found:", len(csv_files))
for file in csv_files:
    print(file.relative_to(data_folder.parent))

CSV files found: 1
data/raw/healthcare_fraud_detection.csv


In [14]:
data_path = data_folder / "raw" / "healthcare_fraud_detection.csv"

df = pd.read_csv(data_path)

df.head()

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0


In [15]:
rows, columns = df.shape

print(f"Number of rows: {rows:,}")
print(f"Number of columns: {columns:,}")

Number of rows: 10,000
Number of columns: 20


In [16]:
# List each column and the data type pandas assigned when reading the CSV.
column_summary = df.dtypes.reset_index()
column_summary.columns = ["Column", "Data Type"]

column_summary

,Column,Data Type
0,Provider_ID,str
1,Claim_ID,str
2,Patient_Age,int64
3,Patient_Gender,str
4,Diagnosis_Code,str
5,Procedure_Code,int64
6,Claim_Amount,float64
7,Approved_Amount,float64
8,Insurance_Type,str
9,Claim_Submission_Date,str


In [17]:
# Count missing values in each column and calculate their percentage of all rows.
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percent": (df.isna().mean() * 100).round(2)
})

missing_summary.sort_values("Missing Count", ascending=False)

,Missing Count,Missing Percent
Provider_Specialty,350,3.5
Prior_Visits_12m,350,3.5
Insurance_Type,350,3.5
Provider_ID,0,0.0
Patient_Gender,0,0.0
Patient_Age,0,0.0
Claim_ID,0,0.0
Diagnosis_Code,0,0.0
Approved_Amount,0,0.0
Claim_Submission_Date,0,0.0


In [18]:
# Check for repeated full records and repeated claim identifiers separately.
duplicate_rows = df.duplicated().sum()
duplicate_claim_ids = df["Claim_ID"].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")
print(f"Repeated Claim IDs: {duplicate_claim_ids:,}")

Duplicate rows: 0
Repeated Claim IDs: 0


In [19]:
# Count each fraud-label value, including missing values, and its share of all claims.
fraud_counts = df["Is_Fraud"].value_counts(dropna=False).sort_index()

fraud_summary = pd.DataFrame({
    "Count": fraud_counts,
    "Percent": (fraud_counts / len(df) * 100).round(2)
})
fraud_summary.index.name = "Is_Fraud"

fraud_summary

,Count,Percent
Is_Fraud,,
0,9171,91.71
1,829,8.29


In [20]:
# Summarize measured quantities, excluding identifiers, codes, and binary flags.
numeric_columns = [
    "Patient_Age",
    "Claim_Amount",
    "Approved_Amount",
    "Days_Between_Service_and_Claim",
    "Number_of_Claims_Per_Provider_Monthly",
    "Length_of_Stay",
    "Prior_Visits_12m"
]

df[numeric_columns].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Patient_Age,10000.0,49.76,17.91,1.00,37.75,50.00,62.00,95.00
Claim_Amount,10000.0,572.80,406.20,60.21,305.20,461.22,711.36,6590.70
Approved_Amount,10000.0,475.51,323.26,50.35,257.20,388.37,598.35,4270.89
Days_Between_Service_and_Claim,10000.0,14.41,8.49,0.00,7.00,14.00,22.00,29.00
Number_of_Claims_Per_Provider_Monthly,10000.0,68.63,14.91,42.00,60.00,66.00,72.00,144.00
Length_of_Stay,10000.0,2.20,1.71,0.00,1.00,2.00,3.00,5.00
Prior_Visits_12m,9650.0,3.03,1.72,0.00,2.00,3.00,4.00,12.00


In [21]:
# Parse dates temporarily to detect invalid entries without changing the original column.
parsed_dates = pd.to_datetime(
    df["Claim_Submission_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

invalid_dates = (
    df["Claim_Submission_Date"].notna() & parsed_dates.isna()
).sum()

# Compare amounts within each claim; separate summary statistics cannot show this.
approved_above_claim = (df["Approved_Amount"] > df["Claim_Amount"]).sum()

print(f"Invalid non-missing dates: {invalid_dates:,}")
print(f"Approved amount exceeds claim amount: {approved_above_claim:,}")
print(f"Date range: {parsed_dates.min()} to {parsed_dates.max()}")

Invalid non-missing dates: 0
Approved amount exceeds claim amount: 0
Date range: 2021-01-06 00:00:00 to 2025-01-26 00:00:00


### Final Data Audit Summary

The dataset was successfully loaded and inspected. It contains
10,000 records and 20 columns.

| Audit check | Finding |
|---|---|
| Missing values | Three columns each have 350 missing values (3.5%): Provider_Specialty, Insurance_Type, and Prior_Visits_12m |
| Exact duplicate rows | 0 |
| Repeated claim IDs | 0 |
| Fraud-label distribution | Label 0: 9,171 (91.71%); label 1: 829 (8.29%) |
| Numerical ranges | No negative values in the seven numerical measurements inspected |
| Date validity | All submission dates parsed successfully |
| Date coverage | January 6, 2021–January 26, 2025 |
| Amount consistency | No approved amount exceeds its claim amount |
